In [2]:
%%capture

import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

from astroquery.sdss import SDSS

```sql
SELECT TOP 10000
    w.w1mpro, w.w2mpro, w.w3mpro,
    (w.w1mpro - w.w2mpro) AS w1_w2_color,
    (w.w2mpro - w.w3mpro) AS w2_w3_color,
    (g.oiii_5007_flux / g.h_beta_flux) AS oiii_hbeta_ratio,
    (g.nii_6584_flux / g.h_alpha_flux) AS nii_halpha_ratio,
    s.class, s.subclass
FROM SpecObj AS s
JOIN galSpecLine AS g ON s.specobjid = g.specobjid
JOIN PhotoTag AS p ON s.bestobjid = p.objid
JOIN wise_xmatch AS x ON p.objid = x.sdss_objid
JOIN wise_allsky AS w ON x.wise_cntr = w.cntr
WHERE s.class = 'GALAXY' 
  AND s.snmedian_g > 40
  AND g.oiii_5007_eqw < 0
  AND g.nii_6584_eqw < 0
  AND g.h_beta_eqw < 0
  AND g.h_alpha_eqw < 0
  AND g.oiii_5007_flux > 5
  AND g.nii_6584_flux > 5
  AND g.h_beta_flux > 5
  AND g.h_alpha_flux > 5
  AND g.oiii_5007_flux > 5 * g.oiii_5007_flux_err
  AND g.nii_6584_flux > 5 * g.nii_6584_flux_err
  AND g.h_beta_flux > 5 * g.h_beta_flux_err
  AND g.h_alpha_flux > 5 * g.h_alpha_flux_err
  AND (2.355 * g.sigma_forbidden) < 500
  AND (2.355 * g.sigma_balmer) < 500
```

In [9]:
query = '''
SELECT TOP 10000
    -- Identification and Astronomy
    s.specobjid, 
    s.objid AS sdss_photoid, 
    s.ra, 
    s.dec, 
    s.z,
    
    -- Task 1a & 3: Diagnostic Emission Line Fluxes (MPA-JHU)
    g.h_beta_flux, 
    g.oiii_5007_flux, 
    g.h_alpha_flux, 
    g.nii_6583_flux, 
    g.sii_6717_flux, 
    g.sii_6731_flux, 
    g.oi_6300_flux,
    
    -- Task 1b: Equivalent Width (WHAN Diagram)
    e.h_alpha_eqw,
    
    -- Task 3: Gas Kinematics ([OIII] velocity dispersion)
    g.oiii_5007_sigma,
    
    -- Task 2: WISE Magnitudes & Match Distance
    w.w1mpro, 
    w.w2mpro, 
    w.w3mpro, 
    w.w4mpro,
    x.match_dist AS wise_distance_arcsec

FROM SpecObjAll AS s
JOIN galSpecLine AS g ON s.specobjid = g.specobjid
JOIN galSpecExtra AS e ON s.specobjid = e.specobjid

-- Official, pre-indexed direct cross-match join
LEFT JOIN wise_xmatch AS x ON s.objid = x.sdss_objid
LEFT JOIN wise_allsky AS w ON x.wise_cntr = w.cntr

WHERE 
    s.z BETWEEN 0.001 AND 0.35          -- Redshift constraint (z < 0.35)
    AND s.class = 'GALAXY'              -- Enforce galaxy targets
    AND s.subClass NOT LIKE '%BROAD%'   -- Exclude Type 1 broad-line objects
    AND g.h_alpha_flux > 0              -- Data sanity filter
    AND (x.match_dist IS NULL OR x.match_dist < 3.0) -- Task 2: Max 3'' matching tolerance

ORDER BY g.h_alpha_flux DESC            -- Top 10,000 brightest targets
'''

In [10]:
# Execute query
result = SDSS.query_sql(query)

# Convert to Pandas for analysis
df = result.to_pandas()

# Verify the count
print(f"Retrieved {len(df)} galaxies.")

InconsistentTableError: Number of header columns (1) inconsistent with data columns in data line 2

```sql
SELECT TOP 20
    g.sii_6717_flux,
    g.sii_6731_flux,
    g.nii_6584_flux,
    g.oi_6300_flux,
    g.oiii_5007_flux,
    g.h_alpha_flux,
    g.h_beta_flux,
    g.h_alpha_eqw,
    g.oiii_sigma,
    w.w1mpro, w.w2mpro, w.w3mpro,
    s.z, s.class, s.subclass, s.bestobjid
FROM SpecObj AS s
    JOIN galSpecLine AS g ON s.specobjid = g.specobjid
    JOIN PhotoTag AS p ON s.bestobjid = p.objid
    LEFT JOIN wise_xmatch AS x ON p.objid = x.sdss_objid
        AND x.match_dist < 3
    LEFT JOIN wise_allsky AS w ON x.wise_cntr = w.cntr 
WHERE s.class = 'GALAXY'
    AND s.snmedian_r > 10.0
    AND s.z < 0.35
    AND g.sii_6717_eqw < 0
    AND g.sii_6731_eqw < 0
    AND g.nii_6584_eqw < 0
    AND g.oi_6300_eqw < 0
    AND g.oiii_5007_eqw < 0
    AND g.h_alpha_eqw < 0
    AND g.h_beta_eqw < 0
    AND (2.355 * g.sigma_forbidden) < 500
    AND (2.355 * g.sigma_balmer) < 500
--    AND g.sii_6717_flux > 3 * g.sii_6717_flux_err
--    AND g.sii_6731_flux > 3 * g.sii_6731_flux_err
--    AND g.nii_6584_flux > 3 * g.nii_6584_flux_err
--    AND g.oi_6300_flux > 3 * g.oi_6300_flux_err
--    AND g.oiii_5007_flux > 3 * g.oiii_5007_flux_err
--    AND g.h_alpha_flux > 3 * g.h_alpha_flux_err
--    AND g.h_beta_flux > 3 * g.h_beta_flux_err
```